# NB08 · Mutaciones: los 24 eventos

Objetivo: demostrar **idempotencia y visibilidad**. No se cronometra qué motor es más rápido -eso ya lo hizo NB04/NB06-.

### Lo que se hereda y no se vuelve a decidir

| | |
|---|---|
| Motor · colección | Qdrant (R03) · `aurum_catalogo__gemini_embedding_2__A4__768` (NB04), 15.000 puntos |
| Escritura síncrona | `QdrantStore.upsert(..., wait=True)` — ya fijado desde NB04, no es D18 |
| Esquema de payload | `PAYLOAD_SCHEMAS["completo"]` (D13), nulos como cadena vacía (D14) |

### Las decisiones de este notebook (`config.yaml` → `nb08_mutaciones`)

| | Decidido | Por qué |
|---|---|---|
| **D18** | Espera activa con timeout, en la **lectura** (la escritura ya es síncrona) | El enunciado (§4.1) exige "saber esperar, fallar o informar" también al leer, no solo al escribir: `wait=True` no garantiza que el HNSW ya lo refleje |
| **D19** | Este notebook se ejecuta **después** de NB07 | NB07 necesitaba calibrar su regla de duplicados contra el catálogo sin mutar — ver `notebooks/07_duplicados.ipynb` |

`eventos_catalogo.csv` trae 24 operaciones ordenadas por `sequence`: 8 actualizaciones (`UPSERT`, el registro ya existía → `catalog_version` 1→2), 8 bajas (`DELETE`) y 8 altas (`UPSERT`, `product_id` nuevo, `catalog_version=1`). La regla de clasificación ya está verificada contra el dato (`README_DATOS.md`), así que `clasificar_eventos` la aplica, no la redescubre.

### 🗺️ Mapa de datos: qué crea cada sección y quién lo usa después

Una variable creada en una celda sigue viva en las de abajo, así que el orden de ejecución importa. La tabla es el hilo conductor: qué produce cada sección y dónde se usa después. Las tres filas en negrita son las que **escriben de verdad en Qdrant**.

| Sección | Crea | Se usa después en |
|---|---|---|
| Setup | `eventos` (24 filas + columna `tipo`) · `completo` (15.000 filas) | B, C, D, E, F |
| A | `indice` (la conexión) · `RECUENTO_ANTES` | `indice`: B→H · `RECUENTO_ANTES`: H |
| B | `eventos_upsert`/`eventos_baja` (el reparto en dos grupos) · `vectores_upsert_por_id` (16 vectores nuevos) · `vectores_baja_por_id` (8 vectores originales, de la cache) | C (construye los puntos) · D (verifica) · F (elige la muestra) |
| **C** | `puntos_upsert`, `ids_baja` · **1ª escritura real en Qdrant** (`indice.upsert`/`indice.delete`, dentro de `aplicar_secuencia`) · `RECUENTO_DESPUES_1` | D, E, F, G, H |
| D | `trazas`, `tabla_trazas` (visibilidad de los 24 eventos) | H |
| E | `tabla_titulos` (título antes/después de las 8 actualizaciones) | H |
| **F** | **2ª escritura**, repite exactamente la de C con los mismos `puntos_upsert`/`ids_baja` · `RECUENTO_DESPUES_2`, `tabla_muestra_idempotencia`, `IDEMPOTENTE` | H |
| **G** | **3ª escritura**, repite la de C con el mismo contenido pero en otro orden · `RECUENTO_BARAJADO` | H |
| H | Junta todo lo anterior en `artifacts/mutaciones.json` | — (es la última celda) |

In [ ]:
# 📄 DATOS · 📚 eventos_catalogo.csv (24) + catalogo_productos.csv (15.000, desde cache)
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from dotenv import load_dotenv

from aurum.almacen import PAYLOAD_SCHEMAS, add_normalized_key, build_payload
from aurum.datos import load_csv
from aurum.embeddings import (
    GeminiEncoder, cache_key, corpus_fingerprint, encode_corpus, truncate_dim,
)
from aurum.motores import CATALOG_PREFIX, catalog_collection_name
from aurum.motores.base import Point
from aurum.motores.qdrant import QdrantStore
from aurum.mutaciones import aplicar_secuencia, clasificar_eventos, verificar_evento

load_dotenv(Path("..") / ".env")
DATA = Path("..") / "data"
CACHE = Path("..") / "artifacts" / "embeddings"
eventos = clasificar_eventos(load_csv(DATA / "eventos_catalogo.csv"))
completo = load_csv(DATA / "catalogo_productos.csv")

MODELO, CONTRATO, PLANTILLA = "gemini-embedding-2", "sin_contrato", "A4"
DIM = 768
COLECCION = catalog_collection_name(model=MODELO, template=PLANTILLA, dim=DIM)
LOTE = 128                    # D15, mismo lote que la ingesta de NB04
ESQUEMA = "completo"          # D13
POLITICA_NULOS = "cadena_vacia"   # D14
NORMALIZACION = "unaccent"    # D03
CAMPOS_FILTRABLES = ["brand", "color"]
TIMEOUT_S, INTERVALO_S = 15.0, 0.5   # D18

print(eventos["tipo"].value_counts().to_string())
print(f"\ncoleccion: {COLECCION}")

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tipo
actualizacion    8
baja             8
alta             8

coleccion: aurum_catalogo__gemini_embedding_2__A4__768


## A · Conexión y recuento inicial

**Entrada:** `COLECCION` (del setup). **Salida:** `indice` -la conexión que usan **todas** las celdas de aquí en adelante- y `RECUENTO_ANTES`.

`RECUENTO_ANTES` no sale de ningún CSV -es lo que Qdrant reporta ahora mismo, en vivo- y es el número contra el que se comparan todos los recuentos posteriores (secciones C, F y G).

In [ ]:
# ⚠️ requiere `make motor-up MOTOR=qdrant`
indice = QdrantStore(
    collection=COLECCION,
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    prefix=CATALOG_PREFIX,
    timeout=30,
)
RECUENTO_ANTES = indice.count()
print(f"puntos antes de los eventos: {RECUENTO_ANTES:,}".replace(",", "."))

puntos antes de los eventos: 15.000


## B · Vectores para altas y actualizaciones

**Entrada:** `eventos` (del setup). **Salida:** `eventos_upsert`/`eventos_baja` (el mismo `eventos`, repartido en dos grupos) y los dos diccionarios de vectores, `vectores_upsert_por_id` y `vectores_baja_por_id` -las claves son `record_id`, así que en las secciones D y F basta con `vectores_..._por_id[record_id]` para encontrar el vector de un evento concreto-.

Las bajas no se reencodean -traen `title`/`brand` vacíos-: solo hace falta su `record_id` para borrarlas, y su vector **original** (de la cache de NB04/NB06) para poder comprobar después que ya no aparecen en una búsqueda. Por eso son dos celdas de código separadas: la primera pide vectores **nuevos** a la API (altas+actualizaciones), la segunda solo **lee de la cache** (bajas) y no gasta ninguna llamada.

In [ ]:
# 📄 DATOS · 📚 16 textos nuevos (8 altas + 8 actualizaciones) — 💸 primera vez: 16 llamadas de pago
eventos_upsert = eventos[eventos["tipo"] != "baja"].reset_index(drop=True)
eventos_baja = eventos[eventos["tipo"] == "baja"].reset_index(drop=True)

_encoder = GeminiEncoder(
    api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
    native_dim=3072, window=8192,
)
codificado_mutados = encode_corpus(
    _encoder, eventos_upsert["text"].tolist(), corpus_id="eventos_catalogo_upsert",
    kind="document", contract=CONTRATO, batch_size=16, cache_dir=CACHE,
)
vectores_upsert_por_id = dict(zip(
    eventos_upsert["record_id"], truncate_dim(codificado_mutados.vectors, DIM)
))
print(f"vectores nuevos: {len(vectores_upsert_por_id)} · desde cache: "
      f"{codificado_mutados.stats.desde_cache}")

vectores nuevos: 16 · desde cache: True


In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) — vectores desde la cache de NB04, solo para las 8 bajas
CORPUS_ID_COMPLETO = f"catalogo_productos__{PLANTILLA}"
from aurum.plantillas import render_template

textos_completo = render_template(completo, PLANTILLA)
clave = cache_key(
    model_id=MODELO, kind="document", contract=CONTRATO,
    corpus_id=CORPUS_ID_COMPLETO, fingerprint=corpus_fingerprint(textos_completo),
)
if not (CACHE / f"{clave}.npy").exists():
    raise RuntimeError(
        f"Los vectores de {CORPUS_ID_COMPLETO} no estan en cache ({clave}).\n"
        f"Deberian estar desde NB04/NB06 -esta celda no paga 15.000 llamadas nuevas."
    )
codificado_completo = encode_corpus(
    _encoder, textos_completo, corpus_id=CORPUS_ID_COMPLETO,
    kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
)
vectores_completo = truncate_dim(codificado_completo.vectors, DIM)
indice_por_record_id = {rid: i for i, rid in enumerate(completo["record_id"])}
vectores_baja_por_id = {
    rid: vectores_completo[indice_por_record_id[rid]] for rid in eventos_baja["record_id"]
}
print(f"vectores originales recuperados para las bajas: {len(vectores_baja_por_id)}")

vectores originales recuperados para las bajas: 8


## C · Construir los puntos y aplicar la secuencia (1ª pasada)

**Entrada:** `eventos_upsert`/`eventos_baja` y los vectores, de la sección B. **Salida:** `puntos_upsert`, `ids_baja` -se reutilizan tal cual en F y G, sin recalcular nada- y `RECUENTO_DESPUES_1`.

El payload de cada punto sale de las columnas de `eventos_catalogo.csv` -`product_id`, `title`, `brand`, `color`, `catalog_version`, `active`, más las claves normalizadas- filtradas por `PAYLOAD_SCHEMAS["completo"]` (D13); el vector es el de la sección B, ya emparejado por `record_id`.

⚠️ **Aquí se escribe de verdad en Qdrant, la primera de tres veces.** `aplicar_secuencia` (`src/aurum/mutaciones.py`) no hace nada propio: envuelve dos llamadas -`indice.upsert(puntos_upsert, ...)` para las 16 altas y actualizaciones, y `indice.delete(record_id)` por cada una de las 8 bajas- para no repetirlas en las tres secciones (C, F, G) que aplican la secuencia completa.

In [ ]:
eventos_upsert_claves = eventos_upsert.copy()
for campo in CAMPOS_FILTRABLES:
    eventos_upsert_claves = add_normalized_key(
        eventos_upsert_claves, field=campo, mode=NORMALIZACION
    )

puntos_upsert = [
    Point(
        record_id=fila["record_id"],
        vector=vectores_upsert_por_id[fila["record_id"]],
        payload=build_payload(
            fila, fields=PAYLOAD_SCHEMAS[ESQUEMA], null_policy=POLITICA_NULOS
        ),
    )
    for fila in eventos_upsert_claves.to_dict("records")
]
ids_baja = list(eventos_baja["record_id"])

# aplicar_secuencia hace, por debajo, exactamente esto:
#   indice.upsert(puntos_upsert, batch_size=LOTE)      -> las 16 altas + actualizaciones
#   for record_id in ids_baja: indice.delete(record_id) -> las 8 bajas
resultado_1 = aplicar_secuencia(indice, puntos_upsert, ids_baja, batch_size=LOTE)
RECUENTO_DESPUES_1 = indice.count()
print(f"aplicados: {resultado_1['n_upsert']} upsert + {resultado_1['n_delete']} delete "
      f"en {resultado_1['segundos']:.2f} s")
print(f"recuento: {RECUENTO_ANTES:,} -> {RECUENTO_DESPUES_1:,} "
      f"(esperado {RECUENTO_ANTES - len(ids_baja) + len(eventos_upsert[eventos_upsert['tipo']=='alta'])})"
      .replace(",", "."))

aplicados: 16 upsert + 8 delete en 0.13 s
recuento: 15.000 -> 15.000 (esperado 15000)


## D · Verificación de visibilidad, evento a evento (D18)

**Entrada:** `eventos` (setup) + los vectores de B -esta sección no escribe nada nuevo en Qdrant, solo **lee** lo que la sección C acaba de escribir-. **Salida:** `tabla_trazas`, que va al artefacto en H.

Las dos rutas que pide el plan -lectura por ID y búsqueda vectorial- para los 24 eventos, no solo una muestra: es barato (24×2 comprobaciones, con reintento hasta 15 s cada una) y es justo lo que va al artefacto.

In [ ]:
trazas = []
for fila in eventos.to_dict("records"):
    tipo = fila["tipo"]
    if tipo == "baja":
        vector = vectores_baja_por_id[fila["record_id"]]
        version_esperada = None
    else:
        vector = vectores_upsert_por_id[fila["record_id"]]
        version_esperada = 2 if tipo == "actualizacion" else None
    traza = verificar_evento(
        indice, tipo, record_id=fila["record_id"], vector=vector,
        catalog_version_esperado=version_esperada,
        timeout_s=TIMEOUT_S, intervalo_s=INTERVALO_S,
    )
    traza["event_id"], traza["product_id"] = fila["event_id"], fila["product_id"]
    trazas.append(traza)

tabla_trazas = pd.DataFrame([
    {
        "event_id": t["event_id"], "tipo": t["tipo"], "product_id": t["product_id"],
        "visible_por_id (lectura directa, get)": t["por_id"]["visible"],
        "seg_hasta_visible_por_id": round(t["por_id"]["segundos"], 3),
        "visible_por_busqueda (indice vectorial, search)": t["por_busqueda"]["visible"],
        "seg_hasta_visible_por_busqueda": round(t["por_busqueda"]["segundos"], 3),
    }
    for t in trazas
])
TODO_VISIBLE = bool(
    tabla_trazas["visible_por_id (lectura directa, get)"].all()
    and tabla_trazas["visible_por_busqueda (indice vectorial, search)"].all()
)
print(f"24/24 eventos visibles por las dos rutas: {TODO_VISIBLE}")
tabla_trazas.style.hide(axis="index")

24/24 eventos visibles por las dos rutas: True


event_id,tipo,product_id,"visible_por_id (lectura directa, get)",seg_hasta_visible_por_id,"visible_por_busqueda (indice vectorial, search)",seg_hasta_visible_por_busqueda
EVT-001,actualizacion,B000G3T55M,True,0.004000,True,0.011000
EVT-002,actualizacion,B07NV4L2W5,True,0.004000,True,0.006000
EVT-003,actualizacion,B00BEFAR80,True,0.005000,True,0.007000
EVT-004,actualizacion,B076HKFZ8N,True,0.002000,True,0.008000
EVT-005,actualizacion,B07JYHSK27,True,0.003000,True,0.006000
EVT-006,actualizacion,B07N379P73,True,0.002000,True,0.009000
EVT-007,actualizacion,B08JCQP3JW,True,0.002000,True,0.008000
EVT-008,actualizacion,B077FZDNJ2,True,0.005000,True,0.006000
EVT-009,baja,B081JP8CC6,True,0.002000,True,0.010000
EVT-010,baja,B07GWRF23V,True,0.002000,True,0.008000


## E · Título nuevo en las actualizaciones

**Entrada:** `completo` (el catálogo original, del setup) y `eventos` -tampoco escribe nada, solo compara tres fuentes de la misma columna `title`-. **Salida:** `tabla_titulos`, al artefacto en H.

D18 ya comprobó `catalog_version=2` evento a evento; falta el segundo requisito del plan: que el `title` leído sea el nuevo, no un punto que exista con el dato viejo. Para verlo de verdad hace falta el **antes** -el título que tenía el producto en `catalogo_productos.csv`, antes de aplicar nada- junto al **después** -lo que devuelve `indice.get()` ahora-, no solo un `True`/`False` agregado.

In [ ]:
titulo_antes_por_id = dict(zip(completo["record_id"], completo["title"]))

tabla_titulos = pd.DataFrame([
    {
        "event_id": fila["event_id"],
        "product_id": fila["product_id"],
        "titulo_antes (catalogo_productos.csv)": titulo_antes_por_id.get(fila["record_id"]),
        "titulo_despues (indice.get)": indice.get(fila["record_id"]).payload.get("title"),
        "titulo_esperado (eventos_catalogo.csv)": fila["title"],
    }
    for fila in eventos[eventos["tipo"] == "actualizacion"].to_dict("records")
])
tabla_titulos["actualizado_ok"] = (
    tabla_titulos["titulo_despues (indice.get)"]
    == tabla_titulos["titulo_esperado (eventos_catalogo.csv)"]
)
print(f"actualizaciones con titulo nuevo: {tabla_titulos['actualizado_ok'].sum()}/8")
tabla_titulos.style.hide(axis="index")

actualizaciones con titulo nuevo: 8/8


event_id,product_id,titulo_antes (catalogo_productos.csv),titulo_despues (indice.get),titulo_esperado (eventos_catalogo.csv),actualizado_ok
EVT-001,B000G3T55M,"NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White 011), S","NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White 011), S - ficha revisada","NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White 011), S - ficha revisada",True
EVT-002,B07NV4L2W5,"Interruptor Universal Inteligente con Wi-Fi, con Control Remoto Meross App. Compatible con Alexa, Google Assistant y SmartThings. Modelo MSS710, Paquete de 4.","Interruptor Universal Inteligente con Wi-Fi, con Control Remoto Meross App. Compatible con Alexa, Google Assistant y SmartThings. Modelo MSS710, Paquete de 4. - ficha revisada","Interruptor Universal Inteligente con Wi-Fi, con Control Remoto Meross App. Compatible con Alexa, Google Assistant y SmartThings. Modelo MSS710, Paquete de 4. - ficha revisada",True
EVT-003,B00BEFAR80,Gel de contacto 250g. axion | Mejora la conductividad de los electrodos TENS y EMS bien usados o de caucho | Gel conductor para electrodos de caucho o parches,Gel de contacto 250g. axion | Mejora la conductividad de los electrodos TENS y EMS bien usados o de caucho | Gel conductor para electrodos de caucho o parches - ficha revisada,Gel de contacto 250g. axion | Mejora la conductividad de los electrodos TENS y EMS bien usados o de caucho | Gel conductor para electrodos de caucho o parches - ficha revisada,True
EVT-004,B076HKFZ8N,"Teka - Campana Extractora, Touch Control y Motor ECOPOWER, Modelo DLH 986, Negro, 67 x 90 x 48 cm","Teka - Campana Extractora, Touch Control y Motor ECOPOWER, Modelo DLH 986, Negro, 67 x 90 x 48 cm - ficha revisada","Teka - Campana Extractora, Touch Control y Motor ECOPOWER, Modelo DLH 986, Negro, 67 x 90 x 48 cm - ficha revisada",True
EVT-005,B07JYHSK27,G-Shock [Casio] de CASIO Frogman 35 Aniversario océano de Magma Solar de Radio GWF-1035F-1JR Hombres,G-Shock [Casio] de CASIO Frogman 35 Aniversario océano de Magma Solar de Radio GWF-1035F-1JR Hombres - ficha revisada,G-Shock [Casio] de CASIO Frogman 35 Aniversario océano de Magma Solar de Radio GWF-1035F-1JR Hombres - ficha revisada,True
EVT-006,B07N379P73,MOMBEBE COSLAND Traje Cocinero Niño Halloween Conjunto de Camiseta Manga Larga 2 Años Blanco,MOMBEBE COSLAND Traje Cocinero Niño Halloween Conjunto de Camiseta Manga Larga 2 Años Blanco - ficha revisada,MOMBEBE COSLAND Traje Cocinero Niño Halloween Conjunto de Camiseta Manga Larga 2 Años Blanco - ficha revisada,True
EVT-007,B08JCQP3JW,"2 Piezas Delantal Infantil Pintura, Bata Impermeable Niño, Blusón Babero con Bolsillos De Manga 3 Larga para 5-10 Niños Dibujar de Arte Escolar, Azul & Rojo (Medio)","2 Piezas Delantal Infantil Pintura, Bata Impermeable Niño, Blusón Babero con Bolsillos De Manga 3 Larga para 5-10 Niños Dibujar de Arte Escolar, Azul & Rojo (Medio) - ficha revisada","2 Piezas Delantal Infantil Pintura, Bata Impermeable Niño, Blusón Babero con Bolsillos De Manga 3 Larga para 5-10 Niños Dibujar de Arte Escolar, Azul & Rojo (Medio) - ficha revisada",True
EVT-008,B077FZDNJ2,"Bedsure Manta Cama 90 Invierno - Manta Sofa Grande Polar Reversible de Franela y Sherpa, Manta Sofa Gruesa 150x200 cm de Microfibra Suave y Borreguito, Gris","Bedsure Manta Cama 90 Invierno - Manta Sofa Grande Polar Reversible de Franela y Sherpa, Manta Sofa Gruesa 150x200 cm de Microfibra Suave y Borreguito, Gris - ficha revisada","Bedsure Manta Cama 90 Invierno - Manta Sofa Grande Polar Reversible de Franela y Sherpa, Manta Sofa Gruesa 150x200 cm de Microfibra Suave y Borreguito, Gris - ficha revisada",True


## F · Repetir la secuencia completa (idempotencia)

**Entrada:** `puntos_upsert`/`ids_baja` de la sección **C** -exactamente los mismos objetos, no se reconstruyen-. **Salida:** `RECUENTO_DESPUES_2`, `tabla_muestra_idempotencia`, `IDEMPOTENTE`, al artefacto en H.

Es la **misma llamada a `aplicar_secuencia` de la sección C**, con los mismos puntos y los mismos IDs a borrar. `upsert` sobrescribe por `record_id` y borrar un id ya ausente no es error en Qdrant, así que el recuento debe quedar igual — pero el recuento total podría cuadrar por casualidad si algo se borrara y otra cosa se creara a la vez. Por eso se captura un punto de cada tipo **antes** y **después** de repetir, y se enseña que ni cambia de contenido ni se duplica.

In [ ]:
muestra_idempotencia = {
    "alta": eventos_upsert.loc[eventos_upsert["tipo"] == "alta", "record_id"].iloc[0],
    "actualizacion": eventos_upsert.loc[
        eventos_upsert["tipo"] == "actualizacion", "record_id"
    ].iloc[0],
    "baja": eventos_baja["record_id"].iloc[0],
}


def _resumen_punto(punto):
    if punto is None:
        return {"existe": False, "catalog_version": None, "title": None}
    return {
        "existe": True,
        "catalog_version": punto.payload.get("catalog_version"),
        "title": punto.payload.get("title"),
    }


antes_de_repetir = {tipo: indice.get(rid) for tipo, rid in muestra_idempotencia.items()}

resultado_2 = aplicar_secuencia(indice, puntos_upsert, ids_baja, batch_size=LOTE)
RECUENTO_DESPUES_2 = indice.count()

despues_de_repetir = {tipo: indice.get(rid) for tipo, rid in muestra_idempotencia.items()}

tabla_muestra_idempotencia = pd.DataFrame([
    {
        "tipo": tipo,
        "record_id": rid,
        **{f"{k}_antes": v for k, v in _resumen_punto(antes_de_repetir[tipo]).items()},
        **{f"{k}_despues": v for k, v in _resumen_punto(despues_de_repetir[tipo]).items()},
    }
    for tipo, rid in muestra_idempotencia.items()
])
def _campos_iguales(a, b):
    # None/NaN no es igual a si mismo con `==` (semantica estandar de NaN):
    # sin este caso, la fila "baja" -None en ambos lados, porque el punto no
    # existe ni antes ni despues- se marcaria como "distinta" por error.
    if pd.isna(a) and pd.isna(b):
        return True
    return a == b


tabla_muestra_idempotencia["identico"] = tabla_muestra_idempotencia.apply(
    lambda f: all(
        _campos_iguales(f[f"{campo}_antes"], f[f"{campo}_despues"])
        for campo in ("existe", "catalog_version", "title")
    ),
    axis=1,
)

IDEMPOTENTE = (
    RECUENTO_DESPUES_1 == RECUENTO_DESPUES_2
    and bool(tabla_muestra_idempotencia["identico"].all())
)
print(f"recuento tras repetir: {RECUENTO_DESPUES_2:,} "
      f"(1a pasada: {RECUENTO_DESPUES_1:,}) · idempotente: {IDEMPOTENTE}".replace(",", "."))
tabla_muestra_idempotencia.style.hide(axis="index")

recuento tras repetir: 15.000 (1a pasada: 15.000) · idempotente: True


tipo,record_id,existe_antes,catalog_version_antes,title_antes,existe_despues,catalog_version_despues,title_despues,identico
alta,b3eb4062-44fd-5e1a-bbdf-1d5357452d74,True,1.000000,Taladro inalámbrico compacto 24 V con dos baterías,True,1.000000,Taladro inalámbrico compacto 24 V con dos baterías,True
actualizacion,e1a0e559-6a49-5be5-b617-ec8a4899e975,True,2.000000,"NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White 011), S - ficha revisada",True,2.000000,"NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White 011), S - ficha revisada",True
baja,096e4fea-09d7-54de-b2a4-3e11a4007cfc,False,nan,nan,False,nan,nan,True


## G · [Opcional] Eventos fuera de orden

**Entrada:** `puntos_upsert`/`ids_baja` de C otra vez, pero barajados. **Salida:** `RECUENTO_BARAJADO`, al artefacto en H.

Tercera y última escritura -misma `aplicar_secuencia`, mismo contenido, solo cambia el orden de envío-. Los 24 eventos tocan 24 `record_id` distintos entre sí, así que el orden no debería alterar el resultado: se baraja y se reaplica, y si el recuento final cambiara, el proceso no sería correcto.

In [ ]:
import random

puntos_barajados = list(puntos_upsert)
random.Random(0).shuffle(puntos_barajados)
ids_baja_barajados = list(ids_baja)
random.Random(1).shuffle(ids_baja_barajados)

aplicar_secuencia(indice, puntos_barajados, ids_baja_barajados, batch_size=LOTE)
RECUENTO_BARAJADO = indice.count()
print(f"recuento tras aplicar fuera de orden: {RECUENTO_BARAJADO:,} · "
      f"igual que antes: {RECUENTO_BARAJADO == RECUENTO_DESPUES_2}".replace(",", "."))

recuento tras aplicar fuera de orden: 15.000 · igual que antes: True


## H · El artefacto

**Entrada:** todo lo de A-G (`RECUENTO_ANTES`, `RECUENTO_DESPUES_1/2`, `RECUENTO_BARAJADO`, `IDEMPOTENTE`, `tabla_muestra_idempotencia`, `tabla_titulos`, `trazas`) — esta celda no calcula nada nuevo, solo empaqueta. **Salida:** `artifacts/mutaciones.json` en disco, fin del notebook.

In [ ]:
artefacto = {
    "coleccion": COLECCION,
    "recuento": {
        "antes": RECUENTO_ANTES,
        "despues_1a_pasada": RECUENTO_DESPUES_1,
        "despues_2a_pasada": RECUENTO_DESPUES_2,
        "despues_fuera_de_orden": RECUENTO_BARAJADO,
        "idempotente": IDEMPOTENTE,
        # via to_json: to_dict() deja bool_/int64 de numpy, que json.dumps no acepta
        "muestra_idempotencia": json.loads(
            tabla_muestra_idempotencia.drop(columns=["record_id"]).to_json(orient="records")
        ),
    },
    "titulos_actualizados_ok": int(tabla_titulos["actualizado_ok"].sum()),
    "eventos": [
        {
            "event_id": t["event_id"], "product_id": t["product_id"], "tipo": t["tipo"],
            "por_id": t["por_id"], "por_busqueda": t["por_busqueda"],
        }
        for t in trazas
    ],
}
destino = Path("..") / "artifacts" / "mutaciones.json"
destino.write_text(json.dumps(artefacto, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Escrito {destino} · {destino.stat().st_size / 1024:.1f} KB")

Escrito ..\artifacts\mutaciones.json · 9.7 KB
